# Calidad de datos · Tablas y catálogo de reglas

## Propósito

Crear las dos tablas del motor declarativo de calidad y cargar el catálogo de reglas.

## Tablas

| Tabla | Grano | Qué responde |
|---|---|---|
| `md_dq_reglas` | una fila por regla | ¿Qué se valida, con qué umbral y qué se hace si no se cumple? |
| `hd_dq_resultados` | una fila por ejecución y regla | ¿Cuántas filas se evaluaron, cuántas incumplen y qué decisión se aplicó? |

## Catálogo

22 reglas (DQ-001 a DQ-022) en las seis dimensiones de calidad: unicidad, completitud, validez, integridad referencial, consistencia y actualidad. Cada regla se define con las columnas del Libro de Gobierno de Datos: regla en lenguaje de negocio, entidad, atributo, dimensión, criticidad, umbral, decisión, justificación del umbral, owner y steward.

Las reglas son declarativas: cada una guarda la consulta que cuenta las filas evaluadas y la que cuenta las filas que incumplen. Para agregar una regla se inserta una fila, sin tocar el código del motor.

## Decisiones ante un incumplimiento

* **DETENER:** el pipeline se detiene. Solo para errores propios del proceso.
* **CUARENTENA:** las filas se aíslan con su motivo, sin eliminarlas, y Gold no las lee. El pipeline sigue y se avisa por Slack.
* **ADVERTIR:** el pipeline sigue y se avisa por Slack.

## Reglas de este notebook

* Se puede correr más de una vez: las tablas usan `CREATE TABLE IF NOT EXISTS` y el catálogo se carga con `MERGE`, sin duplicar.
* `hd_dq_resultados` es histórica: solo se agregan filas.
* Las 22 reglas quedan **activas**. Los umbrales de DQ-019 a DQ-022 se calibraron con los datos reales.
* Al final hay una **prueba en seco** que ejecuta las consultas de las 22 reglas sin guardar resultados ni aislar filas.

## Destino

Esquema de control: `metadata_dataco.supply_chain`.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS metadata_dataco.supply_chain.md_dq_reglas (
  regla_id             STRING        COMMENT 'Identificador de la regla (DQ-nnn)',
  regla_negocio        STRING        COMMENT 'Regla en lenguaje de negocio',
  entidad              STRING        COMMENT 'Tabla sobre la que se evalúa la regla',
  atributo             STRING        COMMENT 'Atributo o atributos que la regla verifica',
  dimension            STRING        COMMENT 'Dimensión de calidad: Unicidad, Completitud, Validez, Integridad referencial, Consistencia o Actualidad',
  criticidad           STRING        COMMENT 'Importancia de la regla: Alta, Media o Baja',
  umbral_cumplimiento  DECIMAL(5,2)  COMMENT 'Porcentaje mínimo de filas que deben cumplir la regla',
  decision             STRING        COMMENT 'Qué hacer si no se cumple: DETENER, CUARENTENA o ADVERTIR',
  justificacion_umbral STRING        COMMENT 'Por qué se eligió ese umbral y esa decisión',
  data_owner           STRING        COMMENT 'Rol responsable del dato',
  steward              STRING        COMMENT 'Rol que revisa y corrige los incumplimientos',
  momento_ejecucion    STRING        COMMENT 'Task que ejecuta la regla: silver o gold',
  consulta_evaluadas   STRING        COMMENT 'Consulta SQL que devuelve el número de filas evaluadas',
  consulta_incumplen   STRING        COMMENT 'Consulta SQL que devuelve el número de filas que incumplen',
  consulta_cuarentena  STRING        COMMENT 'Sentencia SQL que aísla las filas que incumplen. Vacía si Silver ya las aísla',
  estado               STRING        COMMENT 'ACTIVA o INACTIVA. Solo las activas se ejecutan en el pipeline',
  fecha_actualizacion  TIMESTAMP     COMMENT 'Última vez que se cargó o modificó la regla'
)
USING DELTA

In [0]:
%sql
CREATE TABLE IF NOT EXISTS metadata_dataco.supply_chain.hd_dq_resultados (
  id_ejecucion            STRING        COMMENT 'Ejecución del pipeline que evaluó la regla',
  regla_id                STRING        COMMENT 'Regla evaluada (DQ-nnn)',
  momento_ejecucion       STRING        COMMENT 'Task que la ejecutó: silver o gold',
  filas_evaluadas         BIGINT        COMMENT 'Filas sobre las que se evaluó la regla',
  filas_incumplen         BIGINT        COMMENT 'Filas que no cumplen la regla',
  porcentaje_cumplimiento DECIMAL(6,3)  COMMENT 'Porcentaje de filas evaluadas que cumplen',
  umbral_cumplimiento     DECIMAL(5,2)  COMMENT 'Umbral mínimo vigente al momento de la ejecución',
  decision_configurada    STRING        COMMENT 'Decisión definida para la regla: DETENER, CUARENTENA o ADVERTIR',
  resultado               STRING        COMMENT 'CUMPLE, CUARENTENA, ADVERTENCIA, DETENIDO o ERROR',
  filas_aisladas          BIGINT        COMMENT 'Filas que esta ejecución aisló en cuarentena por esta regla',
  detalle                 STRING        COMMENT 'Mensaje de la evaluación o del error',
  fecha_registro          TIMESTAMP     COMMENT 'Momento en que se registró la fila'
)
USING DELTA

In [0]:
%sql
COMMENT ON TABLE metadata_dataco.supply_chain.md_dq_reglas IS
'Catálogo declarativo de reglas de calidad de datos. Grano: una fila por regla (DQ-nnn). Define qué se valida, el umbral, la decisión (DETENER, CUARENTENA o ADVERTIR), el owner y el steward, y guarda las consultas que las evalúan. Maestra: se actualiza al cambiar una regla. Owner: ingeniería de datos.';

COMMENT ON TABLE metadata_dataco.supply_chain.hd_dq_resultados IS
'Resultado de cada regla de calidad por ejecución del pipeline. Grano: una fila por ejecución y regla. Guarda filas evaluadas, filas que incumplen, porcentaje de cumplimiento y la decisión aplicada. Histórica: solo agrega filas. Owner: ingeniería de datos.';

ALTER TABLE metadata_dataco.supply_chain.md_dq_reglas
SET TAGS ('layer' = 'metadata', 'domain' = 'supply_chain', 'owner' = 'ingenieria_de_datos', 'frecuencia' = 'semanal');

ALTER TABLE metadata_dataco.supply_chain.hd_dq_resultados
SET TAGS ('layer' = 'metadata', 'domain' = 'supply_chain', 'owner' = 'ingenieria_de_datos', 'frecuencia' = 'semanal');

## Catálogo de reglas

Las consultas usan cuatro marcadores que el motor reemplaza al ejecutar: `{silver}` (catálogo y esquema de Silver), `{gold}` (Gold), `{control}` (esquema de control) y `{bronze}` (tabla Bronze completa).

Las reglas de unicidad se miden solo sobre filas `VALIDO`, porque las filas con clave inválida quedan en cuarentena con clave 0 y repetirían la clave.

In [0]:
OWNER = "Responsable de Logística"
STEWARD = "Analista de Datos"

PEDIDOS = "{silver}.hd_pedidos"
DETALLE = "{silver}.hd_pedido_detalle"
CLIENTES = "{silver}.md_clientes"
PRODUCTOS = "{silver}.md_productos"
HECHO_PEDIDOS = "{gold}.desc_ft_orders"
RESUMEN = "{gold}.desc_ft_order_summary"
BRONZE = "{bronze}"
CONTROL = "{control}.md_pipeline_control"

ESTADOS_CONOCIDOS = (
    "'COMPLETE', 'PENDING', 'CLOSED', 'PENDING_PAYMENT', 'CANCELED', "
    "'PROCESSING', 'SUSPECTED_FRAUD', 'ON_HOLD', 'PAYMENT_REVIEW'"
)


def contar(tabla, condicion=None):
    donde = f" WHERE {condicion}" if condicion else ""
    return f"SELECT COUNT(*) FROM {tabla}{donde}"


def duplicados(tabla, clave, condicion):
    return f"SELECT COUNT(*) - COUNT(DISTINCT {clave}) FROM {tabla} WHERE {condicion}"


def sin_pareja(tabla, clave, condicion, referencia, clave_referencia, filtro_referencia=None):
    filtro = f" WHERE {filtro_referencia}" if filtro_referencia else ""
    return (
        f"SELECT COUNT(*) FROM (SELECT {clave} FROM {tabla} WHERE {condicion}) origen "
        f"LEFT ANTI JOIN (SELECT DISTINCT {clave_referencia} FROM {referencia}{filtro}) destino "
        f"ON origen.{clave} = destino.{clave_referencia}"
    )


def aislar(tabla, llave, clave, condicion, referencia, clave_referencia, filtro_referencia, motivo):
    filtro = f" WHERE {filtro_referencia}" if filtro_referencia else ""
    return (
        f"MERGE INTO {tabla} AS t USING ("
        f"SELECT origen.{llave} FROM (SELECT {llave}, {clave} FROM {tabla} WHERE estado_calidad = 'VALIDO') origen "
        f"LEFT ANTI JOIN (SELECT DISTINCT {clave_referencia} FROM {referencia}{filtro}) destino "
        f"ON origen.{clave} = destino.{clave_referencia}"
        f") AS s ON t.{llave} = s.{llave} AND t.estado_calidad = 'VALIDO' "
        f"WHEN MATCHED THEN UPDATE SET estado_calidad = 'CUARENTENA', motivo_cuarentena = '{motivo}'"
    )


VOLUMEN = (
    "WITH ventana AS (SELECT next_start_date AS inicio, next_end_date AS fin FROM " + CONTROL + " WHERE pipeline_id = 1), "
    "actual AS (SELECT COUNT(*) / MAX(DATEDIFF(v.fin, v.inicio) + 1) AS promedio FROM " + BRONZE + " b "
    "JOIN ventana v ON b.fecha_pedido_carga BETWEEN v.inicio AND v.fin), "
    "anterior AS (SELECT COUNT(*) / 28.0 AS promedio FROM " + BRONZE + " b "
    "JOIN ventana v ON b.fecha_pedido_carga BETWEEN DATE_SUB(v.inicio, 28) AND DATE_SUB(v.inicio, 1)) "
    "SELECT CASE WHEN anterior.promedio > 0 AND actual.promedio < 0.5 * anterior.promedio THEN 1 ELSE 0 END "
    "FROM actual CROSS JOIN anterior"
)

ACTUALIDAD = (
    "SELECT CASE WHEN EXISTS (SELECT 1 FROM " + BRONZE + " b JOIN " + CONTROL + " c "
    "ON b.fecha_inicio_corte = c.next_start_date AND b.fecha_fin_corte = c.next_end_date "
    "WHERE c.pipeline_id = 1) THEN 0 ELSE 1 END"
)

VALIDO = "estado_calidad = 'VALIDO'"

# (id, regla, entidad, atributo, dimensión, criticidad, umbral, decisión, justificación, momento,
#  evaluadas, incumplen, cuarentena, estado)
REGLAS = [
    ("DQ-001", "Cada pedido aparece una sola vez", "hd_pedidos", "order_id", "Unicidad", "Alta", 100.00, "DETENER",
     "Un pedido repetido después de deduplicar es un error del proceso, no del dato; duplicaría ventas en los indicadores.",
     "silver", contar(PEDIDOS, VALIDO), duplicados(PEDIDOS, "order_id", VALIDO), None, "ACTIVA"),
    ("DQ-002", "Cada línea de pedido aparece una sola vez", "hd_pedido_detalle", "order_item_id", "Unicidad", "Alta", 100.00, "DETENER",
     "Una línea repetida después de deduplicar es un error del proceso; duplicaría ventas y cantidades.",
     "silver", contar(DETALLE, VALIDO), duplicados(DETALLE, "order_item_id", VALIDO), None, "ACTIVA"),
    ("DQ-003", "Cada cliente aparece una sola vez", "md_clientes", "customer_id", "Unicidad", "Alta", 100.00, "DETENER",
     "Un cliente repetido rompe la llave de la dimensión y duplica filas al unir con los pedidos.",
     "silver", contar(CLIENTES, VALIDO), duplicados(CLIENTES, "customer_id", VALIDO), None, "ACTIVA"),
    ("DQ-004", "Cada producto aparece una sola vez", "md_productos", "product_card_id", "Unicidad", "Alta", 100.00, "DETENER",
     "Un producto repetido rompe la llave de la dimensión y duplica filas al unir con el detalle.",
     "silver", contar(PRODUCTOS, VALIDO), duplicados(PRODUCTOS, "product_card_id", VALIDO), None, "ACTIVA"),
    ("DQ-005", "Todo pedido tiene una fecha de pedido válida", "hd_pedidos", "order_date", "Completitud", "Alta", 100.00, "CUARENTENA",
     "Sin fecha no se puede ubicar el pedido en el tiempo ni calcular indicadores; no se descarta, se aísla.",
     "silver", contar(PEDIDOS), contar(PEDIDOS, "order_date IS NULL"), None, "ACTIVA"),
    ("DQ-006", "Todo pedido tiene un cliente informado", "hd_pedidos", "order_customer_id", "Completitud", "Alta", 100.00, "CUARENTENA",
     "Sin cliente no se puede calcular ingresos por cliente; el pedido se aísla para revisión.",
     "silver", contar(PEDIDOS), contar(PEDIDOS, "order_customer_id = 0"), None, "ACTIVA"),
    ("DQ-007", "Todo pedido tiene fecha de envío", "hd_pedidos", "shipping_date", "Completitud", "Media", 99.50, "ADVERTIR",
     "La fecha de envío alimenta Lead Time; una pequeña fracción sin fecha es tolerable, un exceso se avisa.",
     "silver", contar(PEDIDOS), contar(PEDIDOS, "shipping_date IS NULL"), None, "ACTIVA"),
    ("DQ-008", "El riesgo de retraso solo toma el valor 0 o 1", "hd_pedidos", "late_delivery_risk", "Validez", "Alta", 100.00, "CUARENTENA",
     "Un valor fuera del dominio contamina el indicador de riesgo de retraso; el pedido se aísla.",
     "silver", contar(PEDIDOS),
     contar(PEDIDOS, "late_delivery_risk NOT IN (0, 1) OR motivo_cuarentena = 'RIESGO_RETRASO_INVALIDO'"), None, "ACTIVA"),
    ("DQ-009", "La cantidad de cada línea es mayor que cero", "hd_pedido_detalle", "order_item_quantity", "Validez", "Alta", 100.00, "CUARENTENA",
     "Una cantidad no positiva no representa una venta; la línea se aísla.",
     "silver", contar(DETALLE), contar(DETALLE, "order_item_quantity <= 0"), None, "ACTIVA"),
    ("DQ-010", "El estado del pedido pertenece a los valores conocidos", "hd_pedidos", "order_status", "Validez", "Media", 100.00, "ADVERTIR",
     "Un estado nuevo no es necesariamente un error, pero las reglas de negocio de Gold no lo conocen; se avisa.",
     "silver", contar(PEDIDOS), contar(PEDIDOS, f"order_status NOT IN ({ESTADOS_CONOCIDOS})"), None, "ACTIVA"),
    ("DQ-011", "Toda línea de pedido apunta a un pedido existente", "hd_pedido_detalle", "order_id", "Integridad referencial", "Alta", 99.50, "CUARENTENA",
     "Una línea huérfana no se puede asociar a un pedido; se tolera hasta 0,5 % por desfases de carga y se aísla.",
     "silver",
     contar(DETALLE, f"{VALIDO} OR motivo_cuarentena = 'LINEA_SIN_PEDIDO'"),
     sin_pareja(DETALLE, "order_id", f"{VALIDO} OR motivo_cuarentena = 'LINEA_SIN_PEDIDO'", PEDIDOS, "order_id"),
     aislar(DETALLE, "order_item_id", "order_id", VALIDO, PEDIDOS, "order_id", None, "LINEA_SIN_PEDIDO"), "ACTIVA"),
    ("DQ-012", "Toda línea de pedido apunta a un producto existente", "hd_pedido_detalle", "order_item_cardprod_id", "Integridad referencial", "Alta", 99.50, "CUARENTENA",
     "Una línea sin producto válido no se puede clasificar; se tolera hasta 0,5 % y se aísla.",
     "silver",
     contar(DETALLE, f"{VALIDO} OR motivo_cuarentena = 'LINEA_SIN_PRODUCTO'"),
     sin_pareja(DETALLE, "order_item_cardprod_id", f"{VALIDO} OR motivo_cuarentena = 'LINEA_SIN_PRODUCTO'", PRODUCTOS, "product_card_id", VALIDO),
     aislar(DETALLE, "order_item_id", "order_item_cardprod_id", VALIDO, PRODUCTOS, "product_card_id", VALIDO, "LINEA_SIN_PRODUCTO"), "ACTIVA"),
    ("DQ-013", "Todo pedido apunta a un cliente existente", "hd_pedidos", "order_customer_id", "Integridad referencial", "Alta", 99.50, "CUARENTENA",
     "Un pedido sin cliente en la dimensión no entra al indicador de ingresos por cliente; se tolera hasta 0,5 % y se aísla.",
     "silver",
     contar(PEDIDOS, f"{VALIDO} OR motivo_cuarentena = 'PEDIDO_SIN_CLIENTE'"),
     sin_pareja(PEDIDOS, "order_customer_id", f"{VALIDO} OR motivo_cuarentena = 'PEDIDO_SIN_CLIENTE'", CLIENTES, "customer_id", VALIDO),
     aislar(PEDIDOS, "order_id", "order_customer_id", VALIDO, CLIENTES, "customer_id", VALIDO, "PEDIDO_SIN_CLIENTE"), "ACTIVA"),
    ("DQ-014", "Todo hecho de pedido apunta a sus dimensiones", "desc_ft_orders", "cliente_key, location_key, fecha_pedido_key, shipping_mode_key",
     "Integridad referencial", "Alta", 100.00, "DETENER",
     "Un hecho sin su dimensión sale del reporte sin aviso; es un error del proceso de carga de Gold.",
     "gold", contar(HECHO_PEDIDOS, "es_vigente = true"),
     contar(HECHO_PEDIDOS, "es_vigente = true AND (cliente_key IS NULL OR location_key IS NULL OR fecha_pedido_key IS NULL OR shipping_mode_key IS NULL)"),
     None, "ACTIVA"),
    ("DQ-015", "Los pedidos vigentes de Gold son los pedidos válidos y no cancelados de Silver", "desc_ft_orders", "order_id",
     "Consistencia", "Alta", 100.00, "DETENER",
     "Un indicador que no cuadra con la capa limpia es un error; la diferencia en cualquier sentido detiene el proceso.",
     "gold",
     contar(PEDIDOS, f"{VALIDO} AND order_status <> 'CANCELED'"),
     "SELECT (SELECT COUNT(*) FROM (SELECT order_id FROM " + PEDIDOS + " WHERE " + VALIDO + " AND order_status <> 'CANCELED') s "
     "LEFT ANTI JOIN (SELECT order_id FROM " + HECHO_PEDIDOS + " WHERE es_vigente = true) g ON s.order_id = g.order_id) + "
     "(SELECT COUNT(*) FROM (SELECT order_id FROM " + HECHO_PEDIDOS + " WHERE es_vigente = true) g "
     "LEFT ANTI JOIN (SELECT order_id FROM " + PEDIDOS + " WHERE " + VALIDO + " AND order_status <> 'CANCELED') s ON g.order_id = s.order_id)",
     None, "ACTIVA"),
    ("DQ-016", "Cada pedido tiene un solo resumen", "desc_ft_order_summary", "order_id", "Consistencia", "Alta", 100.00, "DETENER",
     "El resumen por pedido es la base única de los indicadores; un pedido repetido duplica sus ventas.",
     "gold", contar(RESUMEN), f"SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM {RESUMEN}", None, "ACTIVA"),
    ("DQ-017", "La fecha de envío no es anterior a la fecha del pedido", "hd_pedidos", "shipping_date, order_date", "Consistencia", "Media", 99.50, "ADVERTIR",
     "Un envío anterior al pedido distorsiona el Lead Time; se tolera una fracción mínima y un exceso se avisa.",
     "silver",
     contar(PEDIDOS, "order_date IS NOT NULL AND shipping_date IS NOT NULL"),
     contar(PEDIDOS, "order_date IS NOT NULL AND shipping_date IS NOT NULL AND CAST(shipping_date AS DATE) < CAST(order_date AS DATE)"),
     None, "ACTIVA"),
    ("DQ-018", "El corte cargado es el que indica la tabla de control", "raw_supply_chain", "fecha_inicio_corte, fecha_fin_corte",
     "Actualidad", "Media", 100.00, "ADVERTIR",
     "El dataset es de 2018, por lo que la frescura se mide contra la ventana de la tabla de control y no contra la fecha de hoy.",
     "silver", "SELECT 1", ACTUALIDAD, None, "ACTIVA"),
    ("DQ-019", "El precio de cada línea es mayor que cero", "hd_pedido_detalle", "order_item_product_price", "Validez", "Baja", 99.00, "ADVERTIR",
     "Umbral calibrado con los datos reales: 0 líneas con precio menor o igual a cero. Se tolera hasta 1 %.",
     "silver", contar(DETALLE, VALIDO), contar(DETALLE, f"{VALIDO} AND order_item_product_price <= 0"), None, "ACTIVA"),
    ("DQ-020", "El descuento de cada línea está entre cero y el monto de la línea", "hd_pedido_detalle", "order_item_discount", "Validez", "Baja", 99.00, "ADVERTIR",
     "Umbral calibrado con los datos reales: 0 líneas con descuento fuera de rango. Se tolera hasta 1 %.",
     "silver", contar(DETALLE, VALIDO),
     contar(DETALLE, f"{VALIDO} AND (order_item_discount < 0 OR order_item_discount > order_item_product_price * order_item_quantity)"),
     None, "ACTIVA"),
    ("DQ-021", "La tasa de descuento está entre 0 y 1", "hd_pedido_detalle", "order_item_discount_rate", "Validez", "Baja", 99.00, "ADVERTIR",
     "Umbral calibrado con los datos reales: 0 líneas con tasa fuera de 0 a 1. Se tolera hasta 1 %.",
     "silver", contar(DETALLE, VALIDO),
     contar(DETALLE, f"{VALIDO} AND (order_item_discount_rate < 0 OR order_item_discount_rate > 1)"), None, "ACTIVA"),
    ("DQ-022", "El volumen diario del corte no cae a menos de la mitad del promedio de las 4 semanas previas", "raw_supply_chain", "fecha_pedido_carga",
     "Actualidad", "Media", 100.00, "ADVERTIR",
     "Una caída fuerte de volumen suele indicar una carga incompleta. Con los cortes reales no hubo caídas bajo el 50 %.",
     "silver", "SELECT 1", VOLUMEN, None, "ACTIVA"),
]

In [0]:
from pyspark.sql import functions as F

columnas = (
    "regla_id string, regla_negocio string, entidad string, atributo string, dimension string, "
    "criticidad string, umbral_cumplimiento double, decision string, justificacion_umbral string, "
    "data_owner string, steward string, momento_ejecucion string, consulta_evaluadas string, "
    "consulta_incumplen string, consulta_cuarentena string, estado string"
)

filas = [
    (r[0], r[1], r[2], r[3], r[4], r[5], r[6], r[7], r[8], OWNER, STEWARD, r[9], r[10], r[11], r[12], r[13])
    for r in REGLAS
]

df_catalogo = (
    spark.createDataFrame(filas, columnas)
    .withColumn("umbral_cumplimiento", F.col("umbral_cumplimiento").cast("decimal(5,2)"))
    .withColumn("fecha_actualizacion", F.current_timestamp())
)
df_catalogo.createOrReplaceTempView("catalogo_dq")

spark.sql("""
MERGE INTO metadata_dataco.supply_chain.md_dq_reglas AS t
USING catalogo_dq AS s
ON t.regla_id = s.regla_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

total = spark.table("metadata_dataco.supply_chain.md_dq_reglas").count()
print(f"✓ Catálogo cargado: {total} reglas en md_dq_reglas")

In [0]:
%sql
SELECT dimension,
       COUNT(*)                                        AS reglas,
       SUM(CASE WHEN decision = 'DETENER'    THEN 1 ELSE 0 END) AS detener,
       SUM(CASE WHEN decision = 'CUARENTENA' THEN 1 ELSE 0 END) AS cuarentena,
       SUM(CASE WHEN decision = 'ADVERTIR'   THEN 1 ELSE 0 END) AS advertir,
       SUM(CASE WHEN estado = 'ACTIVA'       THEN 1 ELSE 0 END) AS activas
FROM metadata_dataco.supply_chain.md_dq_reglas
GROUP BY dimension
ORDER BY dimension

## Prueba en seco

Ejecuta las consultas de las 22 reglas **sin guardar resultados ni aislar filas**. Sirve para comprobar que cada consulta es válida contra las tablas reales y para ver cuántas filas incumplen antes de activar el motor.

* Corre este paso con Silver y Gold ya cargados.
* Si una regla de **DETENER** muestra filas que incumplen, revisa el dato antes de activar el motor: de lo contrario detendría el pipeline en su primera corrida.
* Las reglas DQ-019 a DQ-022 están inactivas; aquí se ejecutan igual para calibrar sus umbrales.

In [0]:
dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")
dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")
dbutils.widgets.text("bronze_catalog", "brz_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "sharepoint", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_supply_chain", "Tabla Bronze")
dbutils.widgets.text("metadata_catalog", "metadata_dataco", "Catálogo Metadata")
dbutils.widgets.text("metadata_schema", "supply_chain", "Esquema Metadata")

MARCADORES = {
    "{silver}": f"{dbutils.widgets.get('silver_catalog')}.{dbutils.widgets.get('silver_schema')}",
    "{gold}": f"{dbutils.widgets.get('gold_catalog')}.{dbutils.widgets.get('gold_schema')}",
    "{control}": f"{dbutils.widgets.get('metadata_catalog')}.{dbutils.widgets.get('metadata_schema')}",
    "{bronze}": f"{dbutils.widgets.get('bronze_catalog')}.{dbutils.widgets.get('bronze_schema')}.{dbutils.widgets.get('bronze_table')}",
}


def resolver(consulta: str) -> str:
    for marcador, valor in MARCADORES.items():
        consulta = consulta.replace(marcador, valor)
    return consulta


resultados = []
for regla in spark.table("metadata_dataco.supply_chain.md_dq_reglas").orderBy("regla_id").collect():
    try:
        evaluadas = spark.sql(resolver(regla["consulta_evaluadas"])).collect()[0][0]
        incumplen = spark.sql(resolver(regla["consulta_incumplen"])).collect()[0][0]
        evaluadas = int(evaluadas or 0)
        incumplen = int(incumplen or 0)
        porcentaje = 100.0 if evaluadas == 0 else round(100.0 * (evaluadas - incumplen) / evaluadas, 3)
        error = None
    except Exception as e:
        evaluadas, incumplen, porcentaje = None, None, None
        error = str(e).splitlines()[0][:200]
    resultados.append((
        regla["regla_id"], regla["decision"], regla["estado"], regla["momento_ejecucion"],
        evaluadas, incumplen, porcentaje, float(regla["umbral_cumplimiento"]), error
    ))

df_seco = spark.createDataFrame(
    resultados,
    "regla_id string, decision string, estado string, momento string, "
    "filas_evaluadas long, filas_incumplen long, porcentaje_cumplimiento double, umbral double, error string"
)
display(df_seco)

con_error = [r for r in resultados if r[8] is not None]
detener_incumplen = [r for r in resultados if r[1] == "DETENER" and r[5]]
print(f"✓ Reglas ejecutadas: {len(resultados) - len(con_error)} de {len(resultados)}")
if con_error:
    print("✗ Reglas con error en la consulta: " + ", ".join(r[0] for r in con_error))
if detener_incumplen:
    print("⚠️ Reglas de DETENER con filas que incumplen: " + ", ".join(r[0] for r in detener_incumplen))
if not con_error and not detener_incumplen:
    print("✓ Todas las consultas son válidas y ninguna regla de DETENER incumple")